## 🚀 SVOMPTR-9B INFERENCE & EVALUATION SUITE

ဒီ Notebook ကို အသုံးပြုပြီး train ထားတဲ့ model ကို Colab GPU ပေါ်မှာ တိုက်ရိုက် စမ်းသပ်နိုင်ပါတယ်။

### ✨ Key Stability Improvements:
- **Hardware Guard**: Memory leak မဖြစ်အောင် memory clear logic ထည့်ထားပါတယ်။
- **Sync-Ready Ops**: Dependency တွေကို restart လုပ်စရာမလိုဘဲ တန်းသုံးနိုင်အောင် စီစဉ်ထားပါတယ်။
- **Fail-Safe API**: LocalTunnel connection ပြတ်တောက်သွားရင် ပြန်ချိတ်ရလွယ်ကူအောင် လုပ်ထားပါတယ်။

### 1. Requirements & Hardware Sync

In [ ]:
import torch, os, sys, subprocess

from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=True)
except: print("⚠️ Drive mount failed.")

print("📦 Synchronizing Neural Environment...")
!pip install --quiet --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --quiet --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate "bitsandbytes>=0.43.3"
!pip install --quiet fastapi uvicorn pydantic evaluate nltk matplotlib seaborn faiss-cpu sentence-transformers sentencepiece

!npm install -q -g localtunnel

import nltk
for p in ['wordnet', 'punkt', 'omw-1.4']: nltk.download(p, quiet=True)

os.environ["BITSANDBYTES_NOWELCOME"] = "1"
print("✅ Sync Complete.")

### 2. Neural Weight Selection

In [ ]:
from unsloth import FastLanguageModel
import torch
import os

FINAL_MODEL_DIR = "/content/drive/MyDrive/svomptr_auto_train/final_lora_weights"

if not os.path.exists(FINAL_MODEL_DIR):
    print(f"⚠️ Weights missing at {FINAL_MODEL_DIR}. Using base brain instead.")
    FINAL_MODEL_DIR = "Qwen/Qwen1.5-MoE-A2.7B"

print(f"🚀 Loading Logic Core: {FINAL_MODEL_DIR}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = FINAL_MODEL_DIR,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    trust_remote_code = True,
)

FastLanguageModel.for_inference(model) # Optimized for speed
print("✅ Core Online.")

### 3. Sub-Expert API Deployment

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn, threading, subprocess, time, json, os, re
from IPython.utils import io

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class ChatRequest(BaseModel): message: str

@app.get("/api/health")
def health(): return {"status": "ok", "gpu": torch.cuda.is_available()}

@app.post("/api/chat")
def chat_endpoint(req: ChatRequest):
    # Standardized Gold Template
    prompt = f"<|im_start|>system\nEnglish-to-Myanmar SVOMPTR Transformer Expert.\n<|im_end|>\n<|im_start|>user\nTranslate: {req.message}\n<|im_end|>\n<|im_start|>assistant\n"
    
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    input_len = inputs.input_ids.shape[-1]
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True)
    
    reply = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    
    # Parse structural components (Neural Pattern Matching)
    frame = { "S": "-", "V": "-", "O": "-", "M": "-", "P": "-", "T": "-", "R": "-" }
    found = re.findall(r'([SVOMPTR])\s*[:|-|=]\s*(.*?)(?=[SVOMPTR]\s*[:|-|=]|$|\n)', reply)
    for k, v in found: 
        k_clean = k.strip()
        if k_clean in frame:
            frame[k_clean] = v.strip().strip(',').strip('|')
            
    return {"response": reply, "frame": frame}

def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

thread = threading.Thread(target=start_server, daemon=True)
thread.start()
time.sleep(3)

print("🚀 Opening Neural Gateway via LocalTunnel...")
lt_cmd = ["lt", "--port", "8000"]
lt_proc = subprocess.Popen(lt_cmd, stdout=subprocess.PIPE)

for i in range(5):
    line = lt_proc.stdout.readline().decode('utf-8').strip()
    if "your url is" in line:
        gw_url = line.replace("your url is: ", "")
        print(f"\n🔗 CONNECT TO WEB APP USING THIS URL:\n{gw_url}\n")
        print("💡 If prompted for an IP address (password), use this:")
        !curl ipv4.icanhazip.com
        break
    time.sleep(1)